# Training a UNET with CoredexMLbench data

In [2]:
import datetime
import pathlib
import sys
import os

In [3]:
import xarray

In [4]:
import matplotlib

In [5]:
import torch.utils.data

In [6]:
# add cnrm code to python path
sys.path.append('/home/users/shaddad/prog/aids_fork/src/cnrm-unet/src')

In [7]:
import data as cnrm_data

In [8]:
#todo set up cordexbench training venv for notebook, add to repo

In [9]:
cordexbench_root = pathlib.Path('/gws/nopw/j04/mohc_shared/cordexbench')
print(cordexbench_root.is_dir())
cordexbench_root

True


PosixPath('/gws/nopw/j04/mohc_shared/cordexbench')

In [10]:
[d1 for d1 in cordexbench_root.iterdir() if d1.is_dir()]


[PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/ALPS_domain'),
 PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/NZ_domain'),
 PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/SA_domain')]

In [11]:
domain_names = {
    'alps': 'ALPS_domain',
    'nz': 'NZ_domain',
    'sa': 'SA_domain',
}

In [12]:
domain_dirs = {domain_key: (cordexbench_root / domain_name) for domain_key,domain_name in domain_names.items()}
domain_dirs

{'alps': PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/ALPS_domain'),
 'nz': PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/NZ_domain'),
 'sa': PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/SA_domain')}

### specifiy training parameters

In [14]:
# Path to the output directory. All outputs will be saved here (model, normalisation stats)
output_dir_path = pathlib.Path('.')

# Path to the training dataset
training_data_path = pathlib.Path('.')

validation_data_path = pathlib.Path('.')

predictors = ['','']
targets = ['','']

referiod_period = {
    'start': datetime.datetime(2021,1,1,0,0),
    'end': datetime.datetime(2022,1,1,0,0),
}

# Loss function. Must be one of 'mse', 'mae', 'emulasym'.
loss = 'mse'
model_channels = 64
channel_mult = [1, 2, 4, 8, 8]
output_channels = 1

# Resolution to use for the input data. Currently, only 16 and 64 are supported.
input_resolution = 16

epochs = 10
batch_size = 32

# Optimiser type: 'adam' or 'sgd' (default: adam)
optimiser_type = 'adam'

learning_rate = 5e-4
scheduler_type = 'onecycle'
scheduler_step_size = 10
scheduler_gamma = 0.1,
scheduler_max_lr = 5e-4

### training the model


In [ ]:
training_dataloader = torch.utils.data.DataLoader(
    training_dataset, batch_size=batch_size, shuffle=True
)
validation_dataloader = torch.utils.data.DataLoader(validation_dataset, batch_size=batch_size)

model = UNet(
    num_2d_predictors=training_dataset.num_2d_predictors,
    num_1d_predictors=training_dataset.num_1d_predictors,
    model_channels=model_channels,
    channel_mult=channel_mult,
    input_resolution=input_resolution,
    output_resolution=output_resolution,
    output_channels=output_channels,
).to(DEVICE)

if loss == "mse":
    loss_fn = torch.nn.MSELoss()
elif loss == "mae":
    loss_fn = torch.nn.L1Loss()
elif loss == "emulasym":
    if "pr" not in targets or len(targets) > 1:
        raise ValueError(
            "EmulASYM loss function should only be used for predicting precipitation (pr)."
        )

    # Fit gamma distributions to the training data
    logger.info("Fitting gamma distributions to the training data.")
    alphas, betas = fit_gamma_distributions(
        training_dataset.target_field,
        training_dataset.time,
        reference_period_start,
        reference_period_end,
    )

    np.save(f"{output_dir_path}/alphas.npy", alphas)
    np.save(f"{output_dir_path}/betas.npy", betas)

    alphas = torch.tensor(alphas, dtype=torch.float32).to(DEVICE)
    betas = torch.tensor(betas, dtype=torch.float32).to(DEVICE)

    loss_fn = EmulASYMLoss(alphas, betas)
else:
    raise ValueError(
        f"Invalid loss function '{loss}'. Must be one of 'mse', 'mae', 'emulasym'."
    )

# Optimiser selection
if optimiser_type.lower() == "adam":
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
elif optimiser_type.lower() == "sgd":
    optimiser = torch.optim.SGD(model.parameters(), lr=learning_rate)
else:
    raise ValueError(f"Unsupported optimiser: {optimiser_type}")

# Scheduler selection
if scheduler_type.lower() == "onecycle":
    scheduler = OneCycleLR(
        optimiser,
        max_lr=scheduler_max_lr,
        steps_per_epoch=len(training_dataloader),
        epochs=epochs,
    )
elif scheduler_type.lower() == "steplr":
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimiser,
        step_size=scheduler_step_size,
        gamma=scheduler_gamma,
    )
else:
    raise ValueError(f"Unsupported scheduler: {scheduler_type}")

best_val_loss = np.inf
for epoch in range(epochs):
    logger.info(f"Epoch {epoch+1}\n-------------------------------")
    train_loop(
        training_dataloader, model, loss_fn, optimiser, scheduler, DEVICE, epoch, scheduler_type
    )

    val_loss = test_loop(validation_dataloader, model, loss_fn, DEVICE, epoch)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = deepcopy(model.state_dict())

    if scheduler is not None:
        logger.info(f"Current learning rate: {scheduler.get_last_lr()[0]}")

    log_prediction_visualisation(model, validation_dataset, DEVICE, epoch, predictors, targets)

model_path = f"{output_dir_path}/model.pth"
logger.info(f"Training complete. Saving model to {model_path}")
torch.save(best_model_state, model_path)

In [29]:
#todo create unet model


In [30]:
#todo define hyperparameters

In [31]:
# todo do data transformations

In [32]:
# todo link to mlflow

In [33]:
#todo create mlflow experiment

In [34]:
# todo setup train stuff 

In [35]:
# todo run training loop

In [36]:
#todo run ionference on model